# 🤖 MyGPT — Train a coding assistant on your GitHub repos (Colab)

This notebook builds a dataset from **your** repositories and fine-tunes a
code model with QLoRA on a **free Colab GPU**, then lets you download the
trained adapter to run locally.

**Before you start:** Runtime ▸ Change runtime type ▸ **GPU (T4)**.

Steps: ① check GPU → ② get the code → ③ install → ④ token → ⑤ pick repos →
⑥ build dataset → ⑦ train → ⑧ test → ⑨ download adapter.


## ① Check the GPU


In [ ]:
!nvidia-smi -L || echo 'No GPU! Set Runtime > Change runtime type > GPU.'


## ② Get the MyGPT code
Clones this repo (the training branch). Public, so no token needed here.


In [ ]:
BRANCH = 'claude/wonderful-hopper-ed8xtj'
!git clone --branch $BRANCH --depth 1 https://github.com/hassanmujtaba22/MyGPT.git
%cd MyGPT


## ③ Install dependencies
Installs the training stack (torch is preinstalled on Colab).


In [ ]:
!pip install -q -r requirements.txt


## ④ Your GitHub token
Create one at https://github.com/settings/tokens with **`repo`** scope to
include private repos. It is entered privately and only used to clone.


In [ ]:
import os, getpass
os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub token (input hidden): ')
GITHUB_USER = 'hassanmujtaba22'  # change if needed


## ⑤ Pick which repos to train on
Leave `REPOS` empty to use **all** your repos, or list specific ones
(comma-separated). Names accept `repo` or `owner/repo`.


In [ ]:
# Examples: 'social-agent-python,cacilian-be,planify-clone'
REPOS = ''  # '' = all repos


## ⑥ Build the coding dataset from your repos
Clones each repo locally (here in Colab), converts source into chat-format
examples, and skips node_modules/build/lockfiles/tests/secrets.


In [ ]:
repos_arg = f"--repos '{REPOS}'" if REPOS.strip() else ''
!python scripts/github_dataset.py --user $GITHUB_USER {repos_arg} \
    --out data/code_train.jsonl
!python scripts/prepare_data.py --config configs/coding.yaml


## ⑦ Fine-tune (QLoRA)
Uses `configs/coding.yaml` (Qwen2.5-Coder). On a free T4, a 1.5–3B model
over a couple of epochs typically finishes in well under an hour.

Tip: for the free T4, the 1.5B coder is the safest fit — uncomment below.


In [ ]:
# !sed -i 's#Qwen2.5-Coder-3B-Instruct#Qwen2.5-Coder-1.5B-Instruct#' configs/coding.yaml
!python scripts/train.py --config configs/coding.yaml


## ⑧ Quick test
Ask your freshly fine-tuned model a coding question.


In [ ]:
import subprocess
print(subprocess.run(
    ['python','-c',
     "import sys; sys.path.insert(0,'scripts');"
     "from _common import load_config; from _llm import get_backend;"
     "cfg=load_config('configs/coding.yaml');"
     "b=get_backend(cfg);"
     "print(b.generate([{'role':'system','content':'You are MyGPT, an expert pair programmer.'},"
     "{'role':'user','content':'Write a TypeScript debounce function.'}], max_new_tokens=256))"],
    capture_output=True, text=True).stdout)


## ⑨ Download your trained adapter
The adapter is small (a few MB). Download it, then on your own machine put
it at `outputs/mygpt-coder-adapter/` and run:
`python scripts/chat.py --config configs/coding.yaml --coding`


In [ ]:
import shutil
shutil.make_archive('mygpt-coder-adapter','zip','outputs/mygpt-coder-adapter')
from google.colab import files
files.download('mygpt-coder-adapter.zip')


---
Made for [MyGPT](https://github.com/hassanmujtaba22/MyGPT). See `docs/coding.md`
for details, and `docs/ollama.md` to run your model via Ollama after merging.
